In [9]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# In a notebook there is no __file__ — hardcode the app dir instead.
APP_DIR = Path.cwd().parent  # microservices/app
sys.path.insert(0, str(APP_DIR))

# Core config (dataset choice, limits)
sys.path.insert(0, str(Path.cwd()))
import config

# All layers (plain dict payloads, see AGENTS.md contract)
from layers_v2.physics import evaluate_physics
from layers_v2.gap import detect_gaps
from layers_v2.ml import FEATURES, add_features, isolation_forest_shap, train_ml_model
from layers_v2.frozen import evaluate_frozen
from layers_v2.forecasting import forecast_reading
from layers_v2.spatial import evaluate_spatial
from layers_v2.fusion import fuse
from layers_v2.aging import evaluate_aging

print("imports OK")

imports OK


In [10]:
# Load the two datasets + station metadata
clean_df = pd.read_parquet(config.CLEAN_PARQUET)
eval_df = pd.read_parquet(config.EVAL_PARQUET)

STATION_COLS = ["lat", "lon", "elevation_m"]

def reading_from_row(row: pd.Series) -> dict:
    """DataFrame row -> L1-style reading dict (AGENTS.md contract)."""
    return {
        "timestamp": str(row["timestamp"]),
        "station_id": row["station_id"],
        "temp_c": row["temp_c"],
        "pressure_hpa": row["pressure_hpa"],
        "humidity_pct": row["humidity_pct"],
    }

def station_from_row(row: pd.Series) -> dict:
    return {col: row[col] for col in STATION_COLS}

# Sorted per-station copies for window-based layers (L3/L4/aging)
clean_df = clean_df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)
eval_df = eval_df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

print("datasets OK:", len(clean_df), "clean /", len(eval_df), "eval rows")


datasets OK: 263160 clean / 263160 eval rows


In [11]:
clean_readings = [reading_from_row(r) for _, r in clean_df.iterrows()]
model, explainer = train_ml_model(clean_readings)
print("L2 model + explainer ready")

L2 model + explainer ready


In [12]:
from chronos import Chronos2Pipeline
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2")
print("Chronos-2 ready")

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 7552.58it/s]

Chronos-2 ready


In [13]:
station = "HYD001"
readings = [reading_from_row(r) for _, r in eval_df.iterrows()]


In [16]:
long_sdf = eval_df[eval_df["station_id"] == station].reset_index(drop=True)
long_readings = [reading_from_row(r) for _, r in long_sdf.iterrows()]
print(forecast_reading(pipeline,long_readings))

{'layer': 'L4', 'station_id': 'HYD001', 'timestamp': '2024-12-31 23:00:00', 'predicted_anomaly': False, 'checks': {'forecast': {'temp_c': {'p5': 21.248, 'p50': 22.175, 'p95': 23.041, 'actual': 22.7, 'severity': 0.0, 'deviation': 0.5254, 'outside_corridor': False}, 'humidity_pct': {'p5': 62.018, 'p50': 65.864, 'p95': 69.672, 'actual': 67.0, 'severity': 0.0, 'deviation': 1.1355, 'outside_corridor': False}, 'pressure_hpa': {'p5': 957.035, 'p50': 957.631, 'p95': 958.357, 'actual': 957.7, 'severity': 0.0, 'deviation': 0.069, 'outside_corridor': False}}}, 'affected_sensors': [], 'reason': None}
